In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv('cleaned_customer_data.csv')

print(f"Loaded cleaned dataset. Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Frequency Feature: Total number of purchases across different channels
df['Frequency'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases'] + df['NumDealsPurchases']

# Monetary Feature: Total amount spent across all product categories
df['Monetary'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']

# Customer Tenure Feature: Calculate how many days the customer has been enrolled (Corrected date format)
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y', errors='coerce')
reference_date = df['Dt_Customer'].max()
df['Customer_Tenure_Days'] = (reference_date - df['Dt_Customer']).dt.days

# Display summary of newly engineered RFM and Tenure features
print("--- RFM & Tenure Features Summary ---")
display(df[['Recency', 'Frequency', 'Monetary', 'Customer_Tenure_Days']].describe())

Loaded cleaned dataset. Shape: 2240 rows, 30 columns

--- RFM & Tenure Features Summary ---


,Recency,Frequency,Monetary,Customer_Tenure_Days
count,2240.000000,2240.000000,2240.000000,2240.000000
mean,49.109375,14.862054,605.798214,353.582143
std,28.962453,7.677173,602.249288,202.122512
min,0.000000,0.000000,5.000000,0.000000
25%,24.000000,8.000000,68.750000,180.750000
50%,49.000000,15.000000,396.000000,355.500000
75%,74.000000,21.000000,1045.500000,529.000000
max,99.000000,44.000000,2525.000000,699.000000


In [3]:
# Import necessary preprocessing libraries
from sklearn.preprocessing import StandardScaler

# Outlier Treatment (Capping extreme outliers using IQR method for key monetary and frequency features)
for col in ['Monetary', 'Income', 'Customer_Tenure_Days']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Cap the outliers
    df[col] = np.where(df[col] > upper_bound, upper_bound, np.where(df[col] < lower_bound, lower_bound, df[col]))

# Selecting features for Clustering (RFM + Tenure + Age + Income)
clustering_features = ['Recency', 'Frequency', 'Monetary', 'Customer_Tenure_Days', 'Age', 'Income']
X = df[clustering_features]

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled array back to a dataframe for verification
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print("--- Feature Scaling & Outlier Treatment Completed ---")
print(f"Shape of scaled feature matrix: {X_scaled_df.shape}")
display(X_scaled_df.head(3))

--- Feature Scaling & Outlier Treatment Completed ---
Shape of scaled feature matrix: (2240, 6)


,Recency,Frequency,Monetary,Customer_Tenure_Days,Age,Income
0,0.307039,1.320826,1.679615,1.531185,0.985345,0.299651
1,-0.383664,-1.154596,-0.961340,-1.190545,1.235733,-0.263808
2,-0.798086,0.799685,0.282733,-0.205773,0.317643,0.943420


In [4]:
# Exporting the scaled features and updated dataframe
output_scaled_path = 'scaled_customer_data.csv'
X_scaled_df.to_csv(output_scaled_path, index=False)

# Save the full processed dataframe
df.to_csv('processed_customer_data.csv', index=False)

print(f"Completed successfully!")
print(f"Scaled dataset saved as '{output_scaled_path}'")
print(f"Full processed dataset saved as 'processed_customer_data.csv'")

Completed successfully!
Scaled dataset saved as 'scaled_customer_data.csv'
Full processed dataset saved as 'processed_customer_data.csv'
